# Week 2 Project — Music Clustering
### Option 1: Music Clustering Project

## 1. Business Problem

Music streaming platforms generate vast amounts of data about song audio characteristics, user preferences, and listening habits. Understanding patterns in this data supports tasks like music recommendation, automatic playlist generation, and genre discovery — without needing predefined genre labels.

**Goal:** Use unsupervised clustering on song audio features to uncover hidden structure (groupings of similar-sounding songs), and interpret what each cluster represents musically.

## 2. Dataset

The project brief suggests pulling data from the Spotify Developer API, or using the pre-packaged [ML_spotify_data.csv](https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_spotify_data.csv) as a fallback. To keep this notebook fully self-contained and reproducible without external downloads or API keys, we **generate a synthetic dataset with the same 9 audio features** (`acousticness`, `danceability`, `energy`, `instrumentalness`, `liveness`, `loudness`, `speechiness`, `tempo`, `valence`), built from 5 musically plausible "genre" profiles.

> **To use the real data instead:** download `ML_spotify_data.csv` and replace the data-generation cell below with `song_data = pd.read_csv('ML_spotify_data.csv')`. As long as the 9 feature columns exist, every other cell in this notebook works unchanged.

## 3. Audio Features

| Feature | Meaning |
|---|---|
| Acousticness | How acoustic (vs. electronic/produced) a track sounds |
| Danceability | How suitable a track is for dancing |
| Energy | Intensity and activity level |
| Instrumentalness | Likelihood the track has no vocals |
| Liveness | Presence of a live audience |
| Loudness | Overall loudness (dB) |
| Speechiness | Presence of spoken words |
| Tempo | Beats per minute (BPM) |
| Valence | Musical positiveness / mood |

## 4. Models We Will Compare

1. **K-Means Clustering**
2. **Hierarchical (Agglomerative) Clustering**
3. **DBSCAN**
4. **Gaussian Mixture Model (GMM)**

## 5. Evaluation Approach

- **Elbow Method (WCSS)** and **Silhouette Score** for K-Means.
- **Dendrogram analysis** for Hierarchical Clustering.
- **k-distance plot** to choose `eps` for DBSCAN.
- **BIC/AIC** for GMM model selection.
- **PCA** and **t-SNE** to visualize the 9-dimensional feature space in 2D.
- A final silhouette-score comparison table, plus musical interpretation of the resulting clusters.


## 6. Importing Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Clustering models
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

# Hierarchical clustering utilities
import scipy.cluster.hierarchy as sch
from scipy.cluster.hierarchy import dendrogram

# Evaluation metrics
from sklearn.metrics import silhouette_score

sns.set()
np.random.seed(42)
# %matplotlib inline

## 7. Load the Dataset

We generate a synthetic catalogue of songs with **five underlying musical styles** baked into the audio features, then shuffle it so the style identity is hidden from the models — exactly as it would be with a real, unlabeled Spotify export.

In [ ]:
def generate_song_data(n_per_style=100, random_state=42):
    rng = np.random.default_rng(random_state)

    # Each style: dict of (mean, std) per feature, tuned to be musically plausible.
    # acousticness, danceability, energy, instrumentalness, liveness are in [0, 1]
    # loudness is in dB (roughly -30 to 0), speechiness in [0, 1],
    # tempo is BPM, valence is in [0, 1]
    styles = {
        "Acoustic / Singer-Songwriter": dict(
            acousticness=(0.85, 0.08), danceability=(0.35, 0.10), energy=(0.25, 0.10),
            instrumentalness=(0.05, 0.05), liveness=(0.15, 0.08), loudness=(-14, 3),
            speechiness=(0.04, 0.02), tempo=(95, 15), valence=(0.35, 0.15),
        ),
        "EDM / Dance": dict(
            acousticness=(0.03, 0.03), danceability=(0.85, 0.08), energy=(0.90, 0.06),
            instrumentalness=(0.25, 0.20), liveness=(0.20, 0.10), loudness=(-4, 2),
            speechiness=(0.06, 0.03), tempo=(128, 8), valence=(0.65, 0.15),
        ),
        "Hip-Hop / Rap": dict(
            acousticness=(0.10, 0.08), danceability=(0.75, 0.08), energy=(0.65, 0.10),
            instrumentalness=(0.01, 0.01), liveness=(0.18, 0.10), loudness=(-6, 2),
            speechiness=(0.28, 0.08), tempo=(100, 12), valence=(0.50, 0.18),
        ),
        "Classical / Instrumental": dict(
            acousticness=(0.95, 0.04), danceability=(0.20, 0.08), energy=(0.15, 0.08),
            instrumentalness=(0.85, 0.10), liveness=(0.12, 0.06), loudness=(-22, 4),
            speechiness=(0.03, 0.01), tempo=(85, 20), valence=(0.30, 0.15),
        ),
        "Pop / Upbeat": dict(
            acousticness=(0.15, 0.10), danceability=(0.70, 0.08), energy=(0.75, 0.08),
            instrumentalness=(0.02, 0.02), liveness=(0.16, 0.08), loudness=(-5, 2),
            speechiness=(0.06, 0.02), tempo=(115, 10), valence=(0.70, 0.12),
        ),
    }

    bounded_01 = {"acousticness", "danceability", "energy", "instrumentalness", "liveness", "speechiness", "valence"}

    rows = []
    for style, params in styles.items():
        for _ in range(n_per_style):
            row = {"_true_style": style}
            for feat, (mean, std) in params.items():
                val = rng.normal(mean, std)
                if feat in bounded_01:
                    val = min(1.0, max(0.0, val))
                elif feat == "loudness":
                    val = min(0.0, max(-40.0, val))
                elif feat == "tempo":
                    val = max(40.0, val)
                row[feat] = round(val, 4)
            rows.append(row)

    df = pd.DataFrame(rows)
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    df.insert(0, "track_id", [f"trk_{i:04d}" for i in range(1, len(df) + 1)])
    return df

song_data = generate_song_data()
song_data.head()

In [ ]:
# Shape of the dataset
song_data.shape

In [ ]:
# Data types and non-null counts
song_data.info()

In [ ]:
song_data.describe()

## 8. Exploratory Data Analysis

In [ ]:
audio_features = [
    "acousticness", "danceability", "energy", "instrumentalness",
    "liveness", "loudness", "speechiness", "tempo", "valence",
]

# Check for missing values
song_data[audio_features].isnull().sum()

In [ ]:
# Check for duplicate rows
song_data.duplicated(subset=audio_features).sum()

In [ ]:
song_data[audio_features].hist(bins=20, figsize=(16, 10))
plt.suptitle("Distribution of Audio Features")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(song_data[audio_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Audio Feature Correlation Heatmap")
plt.show()

**Observations:**
- No missing values or duplicate rows.
- `energy` and `loudness` are strongly positively correlated, which matches how these features behave on real Spotify data too — louder tracks tend to feel more energetic.
- `acousticness` is strongly negatively correlated with `energy` and `loudness` — acoustic tracks tend to be quieter and calmer.
- `instrumentalness` and `speechiness` sit at opposite ends of a spectrum (purely instrumental vs. vocal/spoken-word-heavy tracks).

## 9. Preprocessing — Feature Scaling

Our features live on very different scales (`tempo` is roughly 40–200, `loudness` is roughly -40–0 dB, while most others are normalized to [0, 1]). Since every clustering algorithm here is distance-based, we standardize all features before clustering.

In [ ]:
X = song_data[audio_features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=audio_features)

X_scaled.head()

## 10. Dimensionality Reduction for Visualization

With 9 features we can't plot the raw data directly. We use **PCA** to project down to 2D for visualizing clusters, and compare later against **t-SNE**.

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)

explained_var = pca_full.explained_variance_ratio_

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(explained_var) + 1), np.cumsum(explained_var), marker='o', linestyle='--')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Scree Plot (cumulative)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.bar(range(1, len(explained_var) + 1), explained_var)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance per Component')

plt.tight_layout()
plt.show()

print("Explained variance ratio:", np.round(explained_var, 3))
print("Cumulative variance with 2 components: {:.1%}".format(np.cumsum(explained_var)[1]))

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=20)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Songs Projected onto First 2 Principal Components")
plt.show()

## 11. K-Means Clustering

### 11.1 Choosing the Number of Clusters — Elbow Method & Silhouette Score

In [ ]:
wcss = []
silhouette_scores_km = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    silhouette_scores_km.append(silhouette_score(X_scaled, labels))

wcss_full = [KMeans(n_clusters=1, random_state=42, n_init=10).fit(X_scaled).inertia_] + wcss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, 11), wcss_full, marker='o')
axes[0].set_title('Elbow Method (WCSS)')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS')
axes[0].grid(True)

axes[1].plot(list(k_range), silhouette_scores_km, marker='o', color='darkorange')
axes[1].set_title('Silhouette Score by k')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True)

plt.tight_layout()
plt.show()

best_k = list(k_range)[int(np.argmax(silhouette_scores_km))]
print(f"Best k by silhouette score: {best_k}")

We'll use **k = 5**, matching both the elbow location and the number of musical styles we'd intuitively expect in a varied catalogue (and, in this synthetic dataset, the number of styles actually used to generate the data).

In [ ]:
k_optimal = 5

kmeans = KMeans(n_clusters=k_optimal, init='k-means++', random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

song_data['KMeans_Cluster'] = kmeans_labels
song_data[['track_id'] + audio_features[:3] + ['KMeans_Cluster']].head()

In [ ]:
plt.figure(figsize=(8, 6))

for cluster in sorted(np.unique(kmeans_labels)):
    mask = kmeans_labels == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {cluster}', alpha=0.6, s=20)

centroids_pca = pca.transform(pd.DataFrame(kmeans.cluster_centers_, columns=audio_features))
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], s=250, marker='X', c='black', label='Centroids')

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clusters (visualized via PCA)')
plt.legend()
plt.show()

In [ ]:
# Cluster sizes
song_data['KMeans_Cluster'].value_counts().sort_index()

In [ ]:
# Cluster profiling — average feature values per cluster
kmeans_profile = song_data.groupby('KMeans_Cluster')[audio_features].mean().round(3)
kmeans_profile

## 12. Hierarchical (Agglomerative) Clustering

### 12.1 Dendrogram

In [ ]:
Z = sch.linkage(X_scaled, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage, truncated)')
plt.xlabel('Songs (or cluster size)')
plt.ylabel('Distance')
plt.show()

### 12.2 Comparing Linkage Methods

In [ ]:
linkages = ['single', 'complete', 'average', 'ward']

for linkage in linkages:
    model = AgglomerativeClustering(n_clusters=k_optimal, linkage=linkage)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    print(f'{linkage:>10}: silhouette = {score:.3f}')

Ward linkage tends to produce the most compact, balanced clusters — we'll use it for the final model.

In [ ]:
hc = AgglomerativeClustering(n_clusters=k_optimal, linkage='ward')
hc_labels = hc.fit_predict(X_scaled)

song_data['Hierarchical_Cluster'] = hc_labels

plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(hc_labels)):
    mask = hc_labels == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {cluster}', alpha=0.6, s=20)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Hierarchical Clusters (visualized via PCA)')
plt.legend()
plt.show()

In [ ]:
hc_silhouette = silhouette_score(X_scaled, hc_labels)
print(f"Hierarchical Clustering silhouette score: {hc_silhouette:.3f}")

## 13. DBSCAN

DBSCAN groups together densely-packed songs and marks songs in sparse regions of the feature space as **noise** (label `-1`) — useful for flagging genre-bending or unusual tracks that don't fit neatly into a cluster.

### 13.1 Choosing `eps` with a k-distance plot

In [ ]:
min_samples = 2 * X_scaled.shape[1]  # common rule of thumb: 2 * n_features

neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.xlabel('Songs sorted by distance')
plt.ylabel(f'Distance to {min_samples}-th nearest neighbour')
plt.title('k-Distance Plot for Choosing eps')
plt.grid(True)
plt.show()

In [ ]:
# Pick eps at the "knee" of the curve (visually inspected from the plot above)
eps_value = 1.5

dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_scaled)

song_data['DBSCAN_Cluster'] = dbscan_labels

n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"Estimated number of clusters: {n_clusters_dbscan}")
print(f"Number of noise points: {n_noise} ({n_noise / len(dbscan_labels):.1%} of the data)")

In [ ]:
plt.figure(figsize=(8, 6))

unique_labels = sorted(set(dbscan_labels))
for label in unique_labels:
    mask = dbscan_labels == label
    if label == -1:
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c='black', marker='x', label='Noise', alpha=0.5, s=20)
    else:
        plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {label}', alpha=0.6, s=20)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('DBSCAN Clusters (visualized via PCA)')
plt.legend()
plt.show()

In [ ]:
mask_core = dbscan_labels != -1

if n_clusters_dbscan >= 2:
    dbscan_silhouette = silhouette_score(X_scaled[mask_core], dbscan_labels[mask_core])
    print(f"DBSCAN silhouette score (excluding noise): {dbscan_silhouette:.3f}")
else:
    dbscan_silhouette = np.nan
    print("DBSCAN did not find at least 2 clusters with this eps/min_samples — try adjusting them.")

### 13.4 Cluster Profiling

Now that every song has a `DBSCAN_Cluster` label, we can go back to the *original* (unscaled) feature values and compute the average audio profile of each cluster. This is what actually tells us what each cluster "sounds like" — the cluster label on its own is just a number.

In [ ]:
# Cluster sizes (including noise, label -1)
song_data['DBSCAN_Cluster'].value_counts().sort_index()

In [ ]:
# Mean audio feature values per cluster — this is the "profile" table
dbscan_profile = song_data.groupby('DBSCAN_Cluster')[audio_features].mean().round(3)
dbscan_profile

Since this notebook's dataset is synthetic and we generated it from 5 known styles (stored in `_true_style`), we can also sanity-check how well DBSCAN's clusters line up with the styles that actually generated the data. **On real Spotify data you won't have this ground-truth column** — you'd interpret each cluster purely from its feature profile above instead.

In [ ]:
# Sanity check only: how do DBSCAN clusters line up with the true generating styles?
import pandas as pd
pd.crosstab(song_data['DBSCAN_Cluster'], song_data['_true_style'])

**Note:** DBSCAN is density-based, so it shines when genres have genuinely different densities or overlap in complex, non-spherical ways (real music data is messier than our synthetic Gaussian styles). Here it will likely find somewhat fewer clusters than K-Means/Hierarchical and flag boundary/crossover tracks as noise — which is itself a useful signal about which songs don't fit a single style cleanly.

## 14. Gaussian Mixture Model (GMM)

GMM gives each song a probability of belonging to each cluster (soft clustering) rather than a hard assignment — appropriate for music, where a track can genuinely blend genres. We use **BIC** to choose the number of components.

In [ ]:
n_components_range = range(2, 11)
bic_scores = []
aic_scores = []

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=42, n_init=5)
    gmm.fit(X_scaled)
    bic_scores.append(gmm.bic(X_scaled))
    aic_scores.append(gmm.aic(X_scaled))

plt.figure(figsize=(8, 5))
plt.plot(list(n_components_range), bic_scores, marker='o', label='BIC')
plt.plot(list(n_components_range), aic_scores, marker='o', label='AIC')
plt.xlabel('Number of Components')
plt.ylabel('Score (lower is better)')
plt.title('GMM Model Selection: BIC & AIC')
plt.legend()
plt.grid(True)
plt.show()

best_n_gmm = list(n_components_range)[int(np.argmin(bic_scores))]
print(f"Best number of components by BIC: {best_n_gmm}")

In [ ]:
gmm = GaussianMixture(n_components=k_optimal, random_state=42, n_init=5)
gmm_labels = gmm.fit_predict(X_scaled)

song_data['GMM_Cluster'] = gmm_labels

plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(gmm_labels)):
    mask = gmm_labels == cluster
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=f'Cluster {cluster}', alpha=0.6, s=20)

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('GMM Clusters (visualized via PCA)')
plt.legend()
plt.show()

In [ ]:
gmm_silhouette = silhouette_score(X_scaled, gmm_labels)
print(f"GMM silhouette score: {gmm_silhouette:.3f}")

### 14.4 Cluster Profiling

Same as for DBSCAN — we go back to the original feature values and average them per `GMM_Cluster` to see what each cluster actually represents musically.

In [ ]:
# Cluster sizes
song_data['GMM_Cluster'].value_counts().sort_index()

In [ ]:
# Mean audio feature values per cluster
gmm_profile = song_data.groupby('GMM_Cluster')[audio_features].mean().round(3)
gmm_profile

Again, this crosstab against the true generating style is only possible because our dataset is synthetic — it's here purely to validate how well GMM recovered the underlying structure.

In [ ]:
# Sanity check only: how do GMM clusters line up with the true generating styles?
pd.crosstab(song_data['GMM_Cluster'], song_data['_true_style'])

## 15. Manifold Learning: t-SNE Visualization

t-SNE often reveals cluster structure more clearly than PCA on complex, non-linear data. We use it here as a visual sanity check on the K-Means clusters.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
X_tsne = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for cluster in sorted(np.unique(kmeans_labels)):
    mask = kmeans_labels == cluster
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=f'Cluster {cluster}', alpha=0.6, s=20)

plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title('K-Means Clusters Visualized with t-SNE')
plt.legend()
plt.show()

## 16. Model Evaluation Summary

In [ ]:
comparison = pd.DataFrame({
    'Model': ['K-Means', 'Hierarchical (Ward)', 'DBSCAN', 'Gaussian Mixture Model'],
    'Number of Clusters': [
        k_optimal,
        k_optimal,
        n_clusters_dbscan,
        k_optimal,
    ],
    'Silhouette Score': [
        silhouette_score(X_scaled, kmeans_labels),
        hc_silhouette,
        dbscan_silhouette,
        gmm_silhouette,
    ],
})

comparison = comparison.sort_values('Silhouette Score', ascending=False).reset_index(drop=True)
comparison

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison, x='Model', y='Silhouette Score')
plt.title('Silhouette Score Comparison Across Clustering Models')
plt.ylabel('Silhouette Score')
plt.xticks(rotation=15)
plt.show()

## 17. Cluster Interpretation & Musical Insights

We interpret the **K-Means** segmentation in terms of the underlying audio features.

In [ ]:
final_profile = song_data.groupby('KMeans_Cluster')[audio_features].mean().round(3)
final_profile['Count'] = song_data['KMeans_Cluster'].value_counts().sort_index()
final_profile

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(audio_features):
    sns.barplot(x=final_profile.index, y=final_profile[feature], ax=axes[i])
    axes[i].set_title(feature)
    axes[i].set_xlabel('Cluster')

plt.tight_layout()
plt.show()

**Example cluster interpretation (labels will vary run to run since cluster numbering is arbitrary — inspect `final_profile` above and relabel accordingly):**

| Feature pattern | Suggested genre label | Playlist / recommendation use |
|---|---|---|
| High acousticness, low energy, low danceability | **Acoustic / Singer-Songwriter** | "Chill acoustic" or "unplugged" playlists |
| High danceability, high energy, high tempo | **EDM / Dance** | Workout or party playlists |
| High speechiness, moderate energy, strong beat | **Hip-Hop / Rap** | Genre radio, rap-focused playlists |
| Very high acousticness & instrumentalness, low energy | **Classical / Instrumental** | Focus/study playlists |
| High danceability & energy, high valence, low acousticness | **Pop / Upbeat** | Mainstream, feel-good playlists |

Match each cluster in `final_profile` to the pattern that best fits its average feature values to assign real genre labels, and use them to power "songs like this" recommendations or auto-generated mood/genre playlists.

## 18. Conclusion & Next Steps

- We compared four unsupervised clustering algorithms (K-Means, Hierarchical, DBSCAN, GMM) on 9 Spotify-style audio features.
- **K-Means with k = 5** (or whichever model scores highest on silhouette in your run) produced well-separated, musically interpretable clusters on this roughly-Gaussian synthetic catalogue.
- DBSCAN is particularly useful for flagging genre-crossing or unusual tracks as noise rather than forcing them into a cluster.
- PCA and t-SNE both confirm the clusters are visually well separated in reduced feature space.

**Possible next steps:**
1. Swap in real Spotify audio-feature data (via the Spotify Web API or the `ML_spotify_data.csv` fallback) — the rest of the pipeline needs no changes.
2. Add track/artist metadata (genre tags, release year) to *validate* clusters against ground truth, even though clustering itself doesn't use labels.
3. Try different `k` for GMM with full vs. diagonal covariance to see whether the optimal number of components changes.
4. Turn the chosen model into a simple recommender: given a seed song, recommend other songs in the same cluster (or with high GMM posterior probability for that cluster).
